# Prepare a BioImage Archive submission

Turn an annotation run into a **BioImage Archive (BIA)** submission bundle: MIFA metadata
(`Study`/`Annotations`/`Version` YAML), the BIA file lists (`file_list_images.tsv`,
`file_list_annotations.tsv` with the required `source_image` column), and the image/mask
files, all copied into one self-contained folder.

This is an **optional side branch** of the workflow — independent of training/benchmarking.
If you later publish a model to the BioImage Model Zoo, point its `training_data`/`cite` at
the BIA accession this submission produces (`config.dataset.source_dataset_id`).

> Requires the optional `bia-mifa-models` package (in the `dev`/`mifa` pixi environments).

The cells below run a small self-contained demo. For real use, replace the demo-config cell
with `config = load_config_from_yaml("path/to/your_config.yaml")`.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import tifffile

from omero_annotate_ai.core.annotation_config import (
    AuthorInfo,
    ImageAnnotation,
    create_default_config,
    load_config_from_yaml,  # noqa: F401  (used in real workflows)
)

## 1. Load (or build) your annotation config

In a real workflow: `config = load_config_from_yaml("annotations/<your_config>.yaml")`.

Here we build a small demo config and write a few dummy image/mask tifs to a temp
`output_directory`, so the notebook runs end-to-end with no OMERO server or real data.

In [ ]:
store = Path(tempfile.mkdtemp(prefix="bia_store_"))

config = create_default_config()
config.name = "nuclei_segmentation_demo"
config.study.title = "Nuclei segmentation in U2OS cells"
config.study.description = "Expert-curated nuclear segmentation masks."
config.study.keywords = ["nuclei", "segmentation", "fluorescence"]
config.study.organism = "Homo sapiens"
config.study.funding_statement = "Supported by the Example Funding Council (grant 12345)."
config.dataset.source_dataset_id = "S-BIAD123"
config.dataset.license = "CC-BY-4.0"
config.annotation_methodology.annotation_criteria = "Only in-focus nuclei were annotated."
config.annotation_methodology.annotation_method = "semi_automatic"
config.authors = [AuthorInfo(name="Jane Doe", affiliation="EMBL-EBI")]
config.output.output_directory = store
config.annotations = [
    ImageAnnotation(
        image_id=i, image_name=f"img{i}.tif", annotation_id=f"{i}_0_0",
        category=("training" if i % 2 else "validation"),
        timepoint=0, z_slice=0, channel=0, annotation_type="segmentation_mask",
    )
    for i in (1, 2, 3)
]

# write dummy data into the on-disk layout the pipeline produces (input/ + output/)
(store / "input").mkdir(parents=True, exist_ok=True)
(store / "output").mkdir(parents=True, exist_ok=True)
for ann in config.annotations:
    tifffile.imwrite(store / "input" / f"{ann.annotation_id}.tif",
                     np.zeros((16, 16), dtype="uint8"))
    tifffile.imwrite(store / "output" / f"{ann.annotation_id}_mask.tif",
                     np.zeros((16, 16), dtype="uint8"))
print("config:", config.name, "| annotations:", len(config.annotations))
print("output_directory:", store)

## 2. (Optional) Ensure the data is available locally

The bundle copies the image/mask files from `config.output.output_directory`. If you come
back later with only the config + OMERO data, pull the files down first by composing the
existing pipeline methods — no special cache layer needed:

```python
from omero_annotate_ai.core.annotation_pipeline import create_pipeline
pipeline = create_pipeline(config, conn)
status = pipeline.get_annotation_status_from_disk()      # which masks are already local?
if status["missing"]:                                    # pull what's missing from OMERO
    from omero_annotate_ai.processing.training_functions import prepare_training_data_from_table
    prepare_training_data_from_table(conn, config.omero.table_id, output_dir=config.output.output_directory)
```

In this demo the data is already on disk, so we skip the OMERO round-trip.

In [ ]:
USE_OMERO = False
if USE_OMERO:
    raise NotImplementedError("Set up an OMERO connection and pull missing data here.")
print("Data already local - proceeding to build the bundle.")

## 3. Build the BIA submission bundle

In [ ]:
bundle_dir = Path(tempfile.mkdtemp(prefix="bia_bundle_")) / "submission"
result = config.save_bia_package(bundle_dir, accession="S-BIAD123")

print("Bundle written to:", result["dir"])
print(f"  images: {result['n_images']} | annotations: {result['n_annotations']} "
      f"| files copied: {result['copied']} | missing: {result['missing']}")
print("\nBundle contents:")
for path in sorted(result["dir"].rglob("*")):
    if path.is_file():
        print("  ", path.relative_to(result["dir"]))

## 4. Inspect the file lists and MIFA metadata

In [ ]:
print("=== file_list_annotations.tsv ===")
print(pd.read_csv(result["file_list_annotations"], sep="\t").to_string(index=False))
print("\n=== file_list_images.tsv ===")
print(pd.read_csv(result["file_list_images"], sep="\t").to_string(index=False))
print("\n=== metadata/Annotations_S-BIAD123.yaml ===")
print(result["metadata"]["annotations"].read_text())

## 5. Upload to the BioImage Archive

Submit the `submission/` folder (file lists + data + `metadata/`) to the BioImage Archive
via their FTP/Aspera/Globus transfer, following
[the BIA submission guide](https://www.ebi.ac.uk/bioimage-archive/help-file-list/).
Once published you receive an accession (`S-BIAD###`) and a DOI — feed those back into
`config.dataset.source_dataset_id` / `source_dataset_url` so a Model Zoo export can
reference this dataset as the model's `training_data`.